# SPARC Dark Matter Halo Fitting and ML Benchmark

This notebook builds a full pipeline to:

- download the SPARC rotation-curve dataset
- parse galaxy rotation-curve files
- fit NFW dark matter halos
- build galaxy-level ML features
- train regression models for halo parameters
- generate benchmark plots and summary reports

Two run modes are supported:

- **strict**: minimum 8 usable points per galaxy
- **extended**: minimum 4 usable points per galaxy


## Section 1 — Imports and global settings

In [ ]:
# ============================================================
# SECTION 1: IMPORTS AND GLOBAL SETTINGS
# ============================================================

# If needed in Colab:
# !pip install numpy pandas matplotlib scipy scikit-learn requests tqdm

import os
import re
import glob
import json
import time
import shutil
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from tqdm.auto import tqdm
from scipy.optimize import least_squares
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, WhiteKernel
from sklearn.neural_network import MLPRegressor

# -----------------------------
# USER CONTROLS
# -----------------------------
FORCE_CLEAN_RUN = True
FORCE_REDOWNLOAD = True
USE_SAMPLE_LIMIT = None   # e.g. 20 for quick testing, None for full sample

# Choose one:
RUN_MODE = "strict"       # "strict" or "extended"

# strict   -> min 8 points
# extended -> min 4 points

MODE_CONFIG = {
    "strict": {
        "MIN_POINTS_PER_GALAXY": 8,
        "OUTPUT_TAG": "strict"
    },
    "extended": {
        "MIN_POINTS_PER_GALAXY": 4,
        "OUTPUT_TAG": "extended"
    }
}

if RUN_MODE not in MODE_CONFIG:
    raise ValueError("RUN_MODE must be 'strict' or 'extended'")

MIN_POINTS_PER_GALAXY = MODE_CONFIG[RUN_MODE]["MIN_POINTS_PER_GALAXY"]
OUTPUT_TAG = MODE_CONFIG[RUN_MODE]["OUTPUT_TAG"]

# -----------------------------
# DIRECTORIES
# -----------------------------
DATA_DIR = "/content/sparc_revised_run"
RAW_DIR = os.path.join(DATA_DIR, "raw")
EXTRACT_DIR = os.path.join(RAW_DIR, "extracted")
OUT_DIR = os.path.join(DATA_DIR, f"outputs_{OUTPUT_TAG}")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
LOG_PATH = os.path.join(OUT_DIR, "run_log.txt")
ZIP_PATH = os.path.join(RAW_DIR, "Rotmod_LTG.zip")

URLS = [
    "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1",
    "https://astroweb.case.edu/SPARC/Rotmod_LTG.zip",
    "http://astroweb.cwru.edu/SPARC/Rotmod_LTG.zip",
]

# -----------------------------
# PHYSICS CONSTANTS
# -----------------------------
G = 4.30091e-6       # kpc (km/s)^2 / Msun
rho_crit = 136.0     # Msun / kpc^3
UPS_DISK = 0.5
UPS_BULGE = 0.7
ERROR_FLOOR = 2.0
SAVE_EVERY = 10

print("RUN_MODE =", RUN_MODE)
print("MIN_POINTS_PER_GALAXY =", MIN_POINTS_PER_GALAXY)
print("OUT_DIR =", OUT_DIR)


## Section 2 — Folders and logging

In [ ]:
# ============================================================
# SECTION 2: FOLDERS AND LOGGING
# ============================================================

if FORCE_CLEAN_RUN and os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)

for d in [DATA_DIR, RAW_DIR, EXTRACT_DIR, OUT_DIR, PLOT_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write("SPARC REVISED RUN LOG\n\n")

def log(msg):
    print(msg)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

log("Initialized revised run environment.")
log(f"Run mode: {RUN_MODE}")
log(f"Minimum points per galaxy: {MIN_POINTS_PER_GALAXY}")


## Section 3 — Download and extract dataset

In [ ]:
# ============================================================
# SECTION 3: DOWNLOAD AND EXTRACT DATASET
# ============================================================

def download_file(url, out_path, timeout=180):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)

def download_zip():
    if os.path.exists(ZIP_PATH) and not FORCE_REDOWNLOAD:
        log(f"Using existing zip: {ZIP_PATH}")
        return

    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)

    last_err = None
    for url in URLS:
        try:
            log(f"Trying download: {url}")
            download_file(url, ZIP_PATH)
            size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
            log(f"Downloaded zip successfully. Size = {size_mb:.2f} MB")
            return
        except Exception as e:
            last_err = e
            log(f"Download failed from {url}: {e}")

    raise RuntimeError(f"All downloads failed. Last error: {last_err}")

def extract_zip():
    if os.path.exists(EXTRACT_DIR):
        shutil.rmtree(EXTRACT_DIR)
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    log("Extraction complete.")

download_zip()
extract_zip()


## Section 4 — Find rotmod files

In [ ]:
# ============================================================
# SECTION 4: FIND ROTMOD FILES
# ============================================================

def find_rotmod_files(root_dir):
    files = sorted(glob.glob(os.path.join(root_dir, "**", "*rotmod*.dat"), recursive=True))
    return files

dat_files = find_rotmod_files(EXTRACT_DIR)
log(f"Found {len(dat_files)} rotmod files after extraction.")

if USE_SAMPLE_LIMIT is not None:
    dat_files = dat_files[:USE_SAMPLE_LIMIT]
    log(f"Sample limit active. Using first {len(dat_files)} files.")


## Section 5 — Robust parser

In [ ]:
# ============================================================
# SECTION 5: ROBUST PARSER
# ============================================================

def read_sparc_rotmod(filepath, min_points=MIN_POINTS_PER_GALAXY):
    galaxy_name = os.path.basename(filepath).replace("_rotmod.dat", "").replace(".dat", "")
    rows = []
    distance_mpc = np.nan

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith("#"):
                m = re.search(r"Distance\s*=\s*([0-9.]+)", line)
                if m:
                    distance_mpc = float(m.group(1))
                continue

            parts = line.split()

            numeric_vals = []
            for x in parts:
                try:
                    numeric_vals.append(float(x))
                except:
                    pass

            if len(numeric_vals) >= 6:
                rows.append(numeric_vals[:8] if len(numeric_vals) >= 8 else numeric_vals[:6])

    if len(rows) == 0:
        return None

    max_cols = max(len(r) for r in rows)

    cleaned_rows = []
    for r in rows:
        if len(r) < 6:
            continue
        if len(r) < max_cols:
            r = r + [np.nan] * (max_cols - len(r))
        cleaned_rows.append(r)

    if len(cleaned_rows) == 0:
        return None

    if max_cols >= 8:
        cols = ["Rad", "Vobs", "errV", "Vgas", "Vdisk", "Vbul", "SBdisk", "SBbul"]
        df = pd.DataFrame(cleaned_rows, columns=cols[:max_cols]).iloc[:, :8]
        df.columns = cols
    else:
        cols = ["Rad", "Vobs", "errV", "Vgas", "Vdisk", "Vbul"]
        df = pd.DataFrame(cleaned_rows, columns=cols[:max_cols]).iloc[:, :6]
        df.columns = cols

    df["galaxy"] = galaxy_name
    df["distance_mpc"] = distance_mpc

    keep_cols = ["Rad", "Vobs", "errV", "Vgas", "Vdisk", "Vbul"]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=keep_cols)
    df = df[df["Rad"] > 0].sort_values("Rad").reset_index(drop=True)

    if len(df) < min_points:
        return None

    return df

galaxy_dfs = []
rejected_files = []

for fp in tqdm(dat_files, desc="Parsing galaxy files"):
    df = read_sparc_rotmod(fp, min_points=MIN_POINTS_PER_GALAXY)
    if df is None:
        rejected_files.append(fp)
    else:
        galaxy_dfs.append(df)

log(f"Parsed valid galaxies: {len(galaxy_dfs)}")
log(f"Rejected files: {len(rejected_files)}")

if len(galaxy_dfs) == 0:
    raise RuntimeError("No valid galaxies parsed.")


## Section 6 — Inspect rejected files

In [ ]:
# ============================================================
# SECTION 6: INSPECT REJECTED FILES
# ============================================================

print("Total rejected files:", len(rejected_files))
print("\nFirst 20 rejected filenames:")
for fp in rejected_files[:20]:
    print(os.path.basename(fp))


## Section 7 — Physics helpers

In [ ]:
# ============================================================
# SECTION 7: PHYSICS HELPERS
# ============================================================

def vbar_from_components(df, ups_disk=UPS_DISK, ups_bul=UPS_BULGE):
    term = (
        np.abs(df["Vgas"].values) * df["Vgas"].values +
        ups_disk * np.abs(df["Vdisk"].values) * df["Vdisk"].values +
        ups_bul  * np.abs(df["Vbul"].values)  * df["Vbul"].values
    )
    term = np.clip(term, 0, None)
    return np.sqrt(term)

def r200_from_m200(M200):
    return (3.0 * M200 / (4.0 * np.pi * 200.0 * rho_crit)) ** (1.0 / 3.0)

def nfw_rhos_from_m200_rs(M200, rs):
    R200 = r200_from_m200(M200)
    c = R200 / rs
    f_c = np.log(1 + c) - c / (1 + c)
    return M200 / (4.0 * np.pi * rs**3 * f_c)

def nfw_enclosed_mass(r, M200, rs):
    rho_s = nfw_rhos_from_m200_rs(M200, rs)
    x = np.asarray(r) / rs
    f_x = np.log(1 + x) - x / (1 + x)
    return 4.0 * np.pi * rho_s * rs**3 * f_x

def nfw_velocity(r, M200, rs):
    Menc = nfw_enclosed_mass(r, M200, rs)
    return np.sqrt(G * Menc / np.asarray(r))

def nfw_density(r, M200, rs):
    rho_s = nfw_rhos_from_m200_rs(M200, rs)
    x = np.asarray(r) / rs
    return rho_s / (x * (1 + x)**2)


## Section 8 — Halo fitting function

In [ ]:
# ============================================================
# SECTION 8: HALO FITTING FUNCTION
# ============================================================

def fit_nfw_to_galaxy(df):
    r = df["Rad"].values
    vobs = df["Vobs"].values
    err = np.maximum(df["errV"].values, ERROR_FLOOR)
    vbar = vbar_from_components(df)

    def residuals(theta):
        log10_M200, log10_rs = theta
        M200 = 10**log10_M200
        rs = 10**log10_rs
        vdm = nfw_velocity(r, M200, rs)
        vtot = np.sqrt(vbar**2 + vdm**2)
        return (vtot - vobs) / err

    lower = np.array([9.0, -0.5])
    upper = np.array([14.0, 2.0])
    x0 = np.array([11.5, 1.0])

    result = least_squares(
        residuals,
        x0=x0,
        bounds=(lower, upper),
        method="trf",
        max_nfev=5000
    )

    log10_M200, log10_rs = result.x
    M200 = 10**log10_M200
    rs = 10**log10_rs

    vdm = nfw_velocity(r, M200, rs)
    vtot = np.sqrt(vbar**2 + vdm**2)

    chi2 = np.sum(((vtot - vobs) / err)**2)
    dof = max(len(r) - 2, 1)
    chi2_red = chi2 / dof

    return {
        "galaxy": df["galaxy"].iloc[0],
        "n_points": len(df),
        "distance_mpc": df["distance_mpc"].iloc[0],
        "low_point_fit": bool(len(df) < 8),
        "log10_M200": log10_M200,
        "log10_rs": log10_rs,
        "M200": M200,
        "rs": rs,
        "chi2_red": chi2_red,
        "success": bool(result.success),
        "status": int(result.status),
        "nfev": int(result.nfev),
        "hit_lower_mass": bool(np.isclose(log10_M200, lower[0], atol=1e-3)),
        "hit_upper_mass": bool(np.isclose(log10_M200, upper[0], atol=1e-3)),
        "hit_lower_rs": bool(np.isclose(log10_rs, lower[1], atol=1e-3)),
        "hit_upper_rs": bool(np.isclose(log10_rs, upper[1], atol=1e-3)),
    }

test_fit = fit_nfw_to_galaxy(galaxy_dfs[0])
test_fit


## Section 9 — Run full fit loop

In [ ]:
# ============================================================
# SECTION 9: RUN FULL FIT LOOP
# ============================================================

FIT_CSV = os.path.join(TABLE_DIR, f"nfw_fit_results_{OUTPUT_TAG}.csv")

if os.path.exists(FIT_CSV):
    os.remove(FIT_CSV)

fit_records = []

log("Starting fresh fit table.")
log(f"About to fit {len(galaxy_dfs)} galaxies.")

print("DEBUG: number of galaxies entering fit loop =", len(galaxy_dfs))
print("DEBUG: first 5 galaxy names =", [df["galaxy"].iloc[0] for df in galaxy_dfs[:5]])

fit_start = time.time()
since_save = 0

for df in tqdm(galaxy_dfs, desc="Fitting NFW halos"):
    try:
        rec = fit_nfw_to_galaxy(df)
    except Exception as e:
        rec = {
            "galaxy": df["galaxy"].iloc[0],
            "n_points": len(df),
            "distance_mpc": df["distance_mpc"].iloc[0],
            "low_point_fit": bool(len(df) < 8),
            "log10_M200": np.nan,
            "log10_rs": np.nan,
            "M200": np.nan,
            "rs": np.nan,
            "chi2_red": np.nan,
            "success": False,
            "status": -999,
            "nfev": np.nan,
            "hit_lower_mass": False,
            "hit_upper_mass": False,
            "hit_lower_rs": False,
            "hit_upper_rs": False,
        }
        log(f"Fit failed for {df['galaxy'].iloc[0]}: {e}")

    fit_records.append(rec)
    since_save += 1

    if since_save >= SAVE_EVERY:
        pd.DataFrame(fit_records).to_csv(FIT_CSV, index=False)
        since_save = 0

pd.DataFrame(fit_records).to_csv(FIT_CSV, index=False)

fit_elapsed = time.time() - fit_start

print("DEBUG: fit loop finished")
print("DEBUG: number of fit records =", len(fit_records))
print("DEBUG: fit runtime seconds =", fit_elapsed)

log(f"Fit stage runtime: {fit_elapsed/60:.4f} minutes")


## Section 10 — Load fit results and inspect

In [ ]:
# ============================================================
# SECTION 10: LOAD FIT RESULTS AND INSPECT
# ============================================================

fit_df = pd.read_csv(FIT_CSV)
fit_ok = fit_df[fit_df["success"] == True].copy()

boundary_hits = fit_ok[
    fit_ok["hit_lower_mass"] |
    fit_ok["hit_upper_mass"] |
    fit_ok["hit_lower_rs"] |
    fit_ok["hit_upper_rs"]
].copy()

low_point_fits = fit_ok[fit_ok["low_point_fit"] == True].copy()

log(f"Fit rows saved: {len(fit_df)}")
log(f"Successful fits: {len(fit_ok)}")
log(f"Boundary-hit fits: {len(boundary_hits)}")
log(f"Low-point fits (<8 points): {len(low_point_fits)}")

fit_df.head()


## Section 11 — Build ML features

In [ ]:
# ============================================================
# SECTION 11: BUILD ML FEATURES
# ============================================================

def summarize_galaxy(df):
    r = df["Rad"].values
    vobs = df["Vobs"].values
    err = np.maximum(df["errV"].values, ERROR_FLOOR)
    vgas = np.abs(df["Vgas"].values)
    vdisk = np.abs(df["Vdisk"].values)
    vbul = np.abs(df["Vbul"].values)
    vbar = vbar_from_components(df)

    dm_proxy = np.sqrt(np.clip(vobs**2 - vbar**2, 0, None))

    rec = {
        "galaxy": df["galaxy"].iloc[0],
        "distance_mpc": df["distance_mpc"].iloc[0],
        "n_points": len(df),
        "short_curve_flag": int(len(df) < 8),
        "r_max": np.max(r),
        "r_mean": np.mean(r),
        "vobs_max": np.max(vobs),
        "vobs_mean": np.mean(vobs),
        "vobs_std": np.std(vobs),
        "vbar_max": np.max(vbar),
        "vbar_mean": np.mean(vbar),
        "vgas_max": np.max(vgas),
        "vdisk_max": np.max(vdisk),
        "vbul_max": np.max(vbul),
        "err_mean": np.mean(err),
        "outer_to_inner_v_ratio": np.mean(vobs[-3:]) / max(np.mean(vobs[:3]), 1e-6),
        "bar_to_obs_max_ratio": np.max(vbar) / max(np.max(vobs), 1e-6),
        "dm_proxy_max": np.max(dm_proxy),
        "dm_proxy_mean": np.mean(dm_proxy),
        "inner_slope": (vobs[min(3, len(vobs)-1)] - vobs[0]) / (r[min(3, len(r)-1)] - r[0] + 1e-6),
        "outer_slope": (vobs[-1] - vobs[max(len(vobs)-4, 0)]) / (r[-1] - r[max(len(r)-4, 0)] + 1e-6),
    }
    return rec

all_galaxy_dfs = galaxy_dfs.copy()
all_features_df = pd.DataFrame([summarize_galaxy(df) for df in all_galaxy_dfs])

ml_df = all_features_df.merge(
    fit_ok[["galaxy", "log10_M200", "log10_rs", "chi2_red", "low_point_fit"]],
    on="galaxy",
    how="inner"
).dropna().reset_index(drop=True)

ML_DATA_CSV = os.path.join(TABLE_DIR, f"galaxy_features_and_targets_{OUTPUT_TAG}.csv")
ml_df.to_csv(ML_DATA_CSV, index=False)

log(f"ML dataset rows: {len(ml_df)}")
ml_df.head()


## Section 12 — Train ML models

In [ ]:
# ============================================================
# SECTION 12: TRAIN ML MODELS
# ============================================================

feature_cols = [
    "distance_mpc",
    "n_points",
    "short_curve_flag",
    "r_max",
    "r_mean",
    "vobs_max",
    "vobs_mean",
    "vobs_std",
    "vbar_max",
    "vbar_mean",
    "vgas_max",
    "vdisk_max",
    "vbul_max",
    "err_mean",
    "outer_to_inner_v_ratio",
    "bar_to_obs_max_ratio",
    "dm_proxy_max",
    "dm_proxy_mean",
    "inner_slope",
    "outer_slope"
]

X = ml_df[feature_cols].fillna(0.0).values
y_mass = ml_df["log10_M200"].values
y_rs = ml_df["log10_rs"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "GaussianProcess": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianProcessRegressor(
            kernel=C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2)) + WhiteKernel(1.0),
            normalize_y=True,
            random_state=42
        ))
    ]),
    "NeuralNetwork": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=3000,
            random_state=42
        ))
    ])
}

def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

ml_metrics = []
pred_store = {}

for name, model in models.items():
    pred_mass = cross_val_predict(model, X, y_mass, cv=kf)
    pred_rs = cross_val_predict(model, X, y_rs, cv=kf)

    m1 = metrics(y_mass, pred_mass)
    m2 = metrics(y_rs, pred_rs)

    ml_metrics.append({"model": name, "target": "log10_M200", **m1})
    ml_metrics.append({"model": name, "target": "log10_rs", **m2})

    pred_store[(name, "mass")] = pred_mass
    pred_store[(name, "rs")] = pred_rs

ml_metrics_df = pd.DataFrame(ml_metrics)
ML_METRICS_CSV = os.path.join(TABLE_DIR, f"ml_metrics_5fold_{OUTPUT_TAG}.csv")
ml_metrics_df.to_csv(ML_METRICS_CSV, index=False)

ml_metrics_df


## Section 13 — Fit final NN models for prediction plots

In [ ]:
# ============================================================
# SECTION 13: FIT FINAL MODELS FOR PREDICTION PLOTS
# ============================================================

mass_model = models["NeuralNetwork"]
rs_model = models["NeuralNetwork"]

mass_model.fit(X, y_mass)
rs_model.fit(X, y_rs)

ml_df["pred_log10_M200"] = mass_model.predict(X)
ml_df["pred_log10_rs"] = rs_model.predict(X)

ML_PRED_CSV = os.path.join(TABLE_DIR, f"ml_predictions_full_{OUTPUT_TAG}.csv")
ml_df.to_csv(ML_PRED_CSV, index=False)

ml_df.head()


## Section 14 — Plot sample rotation curves

In [ ]:
# ============================================================
# SECTION 14: PLOT SAMPLE ROTATION CURVES
# ============================================================

name_to_df = {df["galaxy"].iloc[0]: df for df in all_galaxy_dfs}
name_to_fit = {row["galaxy"]: row for _, row in fit_ok.iterrows()}

def plot_sample_rotation_curves(n=9):
    sample_names = fit_ok.sort_values("chi2_red").head(n)["galaxy"].tolist()
    ncols = 3
    nrows = int(np.ceil(len(sample_names) / ncols))
    plt.figure(figsize=(5 * ncols, 4 * nrows))

    for i, name in enumerate(sample_names, 1):
        df = name_to_df[name]
        r = df["Rad"].values
        vobs = df["Vobs"].values
        err = df["errV"].values
        vbar = vbar_from_components(df)

        plt.subplot(nrows, ncols, i)
        plt.errorbar(r, vobs, yerr=err, fmt='o', ms=3, label="Observed")
        plt.plot(r, vbar, lw=2, label="Baryons")
        plt.plot(r, np.abs(df["Vgas"].values), '--', lw=1, label="Gas")
        plt.plot(r, np.abs(df["Vdisk"].values), '--', lw=1, label="Disk")
        if np.any(np.abs(df["Vbul"].values) > 0):
            plt.plot(r, np.abs(df["Vbul"].values), '--', lw=1, label="Bulge")

        plt.title(name)
        plt.xlabel("Radius [kpc]")
        plt.ylabel("Velocity [km/s]")
        plt.grid(alpha=0.3)
        if i == 1:
            plt.legend(fontsize=8)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, f"sample_rotation_curves_{OUTPUT_TAG}.png")
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

plot_sample_rotation_curves()


## Section 15 — Plot fitted decomposition

In [ ]:
# ============================================================
# SECTION 15: PLOT FITTED DECOMPOSITION
# ============================================================

def plot_fit_grid(n=9):
    sample_names = fit_ok.sort_values("chi2_red").head(n)["galaxy"].tolist()
    ncols = 3
    nrows = int(np.ceil(len(sample_names) / ncols))
    plt.figure(figsize=(5 * ncols, 4 * nrows))

    for i, name in enumerate(sample_names, 1):
        df = name_to_df[name]
        fr = name_to_fit[name]
        r = df["Rad"].values
        vobs = df["Vobs"].values
        err = np.maximum(df["errV"].values, ERROR_FLOOR)
        vbar = vbar_from_components(df)
        vdm = nfw_velocity(r, 10**fr["log10_M200"], 10**fr["log10_rs"])
        vtot = np.sqrt(vbar**2 + vdm**2)

        plt.subplot(nrows, ncols, i)
        plt.errorbar(r, vobs, yerr=err, fmt='o', ms=3, label="Observed")
        plt.plot(r, vbar, lw=2, label="Baryons")
        plt.plot(r, vdm, lw=2, label="Dark matter")
        plt.plot(r, vtot, lw=3, label="Total fit")
        plt.title(f"{name}\nlogM={fr['log10_M200']:.2f}, logrs={fr['log10_rs']:.2f}, chi2r={fr['chi2_red']:.2f}")
        plt.xlabel("Radius [kpc]")
        plt.ylabel("Velocity [km/s]")
        plt.grid(alpha=0.3)
        if i == 1:
            plt.legend(fontsize=8)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, f"nfw_fit_grid_{OUTPUT_TAG}.png")
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

plot_fit_grid()


## Section 16 — Plot ML predicted vs actual

In [ ]:
# ============================================================
# SECTION 16: PLOT ML PREDICTED VS ACTUAL
# ============================================================

def plot_ml_pred_vs_actual():
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.scatter(ml_df["log10_M200"], ml_df["pred_log10_M200"], s=18)
    a = min(ml_df["log10_M200"].min(), ml_df["pred_log10_M200"].min())
    b = max(ml_df["log10_M200"].max(), ml_df["pred_log10_M200"].max())
    plt.plot([a, b], [a, b], '--')
    plt.xlabel("Actual log10_M200")
    plt.ylabel("Predicted log10_M200")
    plt.title("Halo mass")
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.scatter(ml_df["log10_rs"], ml_df["pred_log10_rs"], s=18)
    a = min(ml_df["log10_rs"].min(), ml_df["pred_log10_rs"].min())
    b = max(ml_df["log10_rs"].max(), ml_df["pred_log10_rs"].max())
    plt.plot([a, b], [a, b], '--')
    plt.xlabel("Actual log10_rs")
    plt.ylabel("Predicted log10_rs")
    plt.title("Halo scale radius")
    plt.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, f"ml_pred_vs_actual_{OUTPUT_TAG}.png")
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

plot_ml_pred_vs_actual()


## Section 17 — Plot density profiles

In [ ]:
# ============================================================
# SECTION 17: PLOT DENSITY PROFILES
# ============================================================

def plot_density_profiles(n=4):
    sample_names = fit_ok.sort_values("chi2_red").head(n)["galaxy"].tolist()
    plt.figure(figsize=(10, 8))

    for i, name in enumerate(sample_names, 1):
        fr = name_to_fit[name]
        r = np.logspace(-1, 2, 250)
        rho = nfw_density(r, 10**fr["log10_M200"], 10**fr["log10_rs"])

        plt.subplot(2, 2, i)
        plt.loglog(r, rho, lw=2)
        plt.xlabel("Radius [kpc]")
        plt.ylabel(r"Density [M$_\odot$/kpc$^3$]")
        plt.title(name)
        plt.grid(alpha=0.3, which="both")

    plt.tight_layout()
    path = os.path.join(PLOT_DIR, f"density_profiles_{OUTPUT_TAG}.png")
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

plot_density_profiles()


## Section 18 — Plot with vs without dark matter

In [ ]:
# ============================================================
# SECTION 18: PLOT WITH VS WITHOUT DARK MATTER
# ============================================================

def plot_with_without_dm():
    name = fit_ok.sort_values("chi2_red").iloc[len(fit_ok)//2]["galaxy"]
    df = name_to_df[name]
    fr = name_to_fit[name]

    r = df["Rad"].values
    vobs = df["Vobs"].values
    err = np.maximum(df["errV"].values, ERROR_FLOOR)
    vbar = vbar_from_components(df)
    vdm = nfw_velocity(r, 10**fr["log10_M200"], 10**fr["log10_rs"])
    vtot = np.sqrt(vbar**2 + vdm**2)

    plt.figure(figsize=(8, 5))
    plt.errorbar(r, vobs, yerr=err, fmt='o', label="Observed")
    plt.plot(r, vbar, lw=3, label="Without dark matter")
    plt.plot(r, vtot, lw=3, label="With dark matter")
    plt.plot(r, vdm, lw=2, label="Dark matter only")
    plt.xlabel("Radius [kpc]")
    plt.ylabel("Velocity [km/s]")
    plt.title(f"{name}: with vs without dark matter")
    plt.grid(alpha=0.3)
    plt.legend()

    path = os.path.join(PLOT_DIR, f"{name}_with_without_dm_{OUTPUT_TAG}.png")
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

plot_with_without_dm()


## Section 19 — Summary report

In [ ]:
# ============================================================
# SECTION 19: SUMMARY REPORT
# ============================================================

summary = {
    "run_mode": RUN_MODE,
    "min_points_per_galaxy": int(MIN_POINTS_PER_GALAXY),
    "found_rotmod_files": int(len(dat_files)),
    "parsed_valid_galaxies": int(len(all_galaxy_dfs)),
    "fit_rows_saved": int(len(fit_df)),
    "successful_fits": int(len(fit_ok)),
    "boundary_hit_fits": int(len(boundary_hits)),
    "low_point_fits": int(len(low_point_fits)),
    "median_log10_M200": float(fit_ok["log10_M200"].median()),
    "median_log10_rs": float(fit_ok["log10_rs"].median()),
    "median_chi2_red": float(fit_ok["chi2_red"].median()),
    "runtime_minutes_fit_stage": float(fit_elapsed / 60.0),
}

for _, row in ml_metrics_df.iterrows():
    summary[f"{row['model']}_{row['target']}_MAE"] = float(row["MAE"])
    summary[f"{row['model']}_{row['target']}_RMSE"] = float(row["RMSE"])
    summary[f"{row['model']}_{row['target']}_R2"] = float(row["R2"])

SUMMARY_JSON = os.path.join(TABLE_DIR, f"summary_report_{OUTPUT_TAG}.json")
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

log("Final summary:")
log(json.dumps(summary, indent=2))

summary


## Section 20 — Output file locations

In [ ]:
# ============================================================
# SECTION 20: OUTPUT FILE LOCATIONS
# ============================================================

print("Outputs directory:", OUT_DIR)

print("\nTables:")
for fp in sorted(glob.glob(os.path.join(TABLE_DIR, "*"))):
    print(fp)

print("\nPlots:")
for fp in sorted(glob.glob(os.path.join(PLOT_DIR, "*.png"))):
    print(fp)

print("\nLog file:")
print(LOG_PATH)
